# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

print('rows:', len(df))
print('total revenue:', df['revenue'].sum())
print('total units:', df['qty'].sum())

rows: 400
total revenue: 8520.0
total units: 783


Explanation: I made a revenue column by multiplying quantity by price for each order, then added up both columns. The 400 orders brought in 8520.0 dollars across 783 units.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')[['revenue']].sum()
by_category['percent'] = round(by_category['revenue'] / df['revenue'].sum() * 100, 2)
by_category = by_category.sort_values('revenue', ascending=False)

by_category

,revenue,percent
category,,
Food,4293.0,50.39
Merch,1771.5,20.79
Drink,1554.0,18.24
RainGear,901.5,10.58


Explanation: I grouped the orders by category, added up the revenue in each one, and divided by the overall total to get each category's percent share. Food brought in the most at 4293.0 dollars (around 50 percent), then Merch at 1771.5 dollars (around 21 percent), Drink at 1554.0 dollars (around 18 percent), and RainGear at 901.5 dollars (around 11 percent).

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendors = round(df.groupby('vendor_id')[['revenue']].mean(), 2)
vendors['orders'] = df.groupby('vendor_id')['revenue'].count()
vendors = vendors.sort_values('revenue', ascending=False)

vendors

,revenue,orders
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


Explanation: I grouped the orders by vendor, took the average revenue per order, and added a count column next to it. V-01 is highest at 22.60 dollars on 94 orders, V-18 at 21.75 on 108 orders, V-05 at 20.58 on 93 orders, and V-10 at 20.31 on 105 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch = df[df['category'] == 'Merch']['revenue'].sum()
share = round(merch / df['revenue'].sum() * 100, 1)

print('Merch revenue:', merch)
print('Merch share:', share, 'percent')

Merch revenue: 1771.5
Merch share: 20.8 percent


Explanation: I pulled out just the Merch rows, added up their revenue, and divided by the overall total to get the share. Merch brought in 1771.5 dollars, which is 20.8 percent of all revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

rows_before = len(df)
revenue_before = df['revenue'].sum()

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

rows_after = len(joined)
revenue_after = joined['revenue'].sum()

print('rows before:', rows_before)
print('rows after:', rows_after)
print('revenue before:', revenue_before)
print('revenue after:', revenue_after)

missing = joined[joined['vendor_name'].isna()]

print('unmatched vendor:', missing['vendor_id'].unique())
print('unmatched rows:', len(missing))
print('unmatched revenue:', missing['revenue'].sum())

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown')

joined.head()

rows before: 400
rows after: 400
revenue before: 8520.0
revenue after: 8520.0
unmatched vendor: ['V-18']
unmatched rows: 108
unmatched revenue: 2349.0


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown
2,V-18,Drink,3,4.5,13.5,Unknown
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown


**The unmatched vendor, and what I did about it:** V-18 is not in the lookup table, so 108 of the 400 orders came back with a blank vendor name. I filled those blanks with Unknown instead of dropping the rows, since removing them would make the totals wrong.

Explanation: The row count was 400 and revenue was 8520.0, which means nothing was lost or double counted. V-18 is not in the lookup, so 108 orders came back blank and I filled them in as Unknown.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


Explanation: I made a pivot table with vendors down the side, categories across the top, and revenue in each cell, and the totals at the bottom. Every vendor makes most of its money on Food, and the four vendors end up close in total revenue at around 2000 dollars even when their category mixes are different.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) For the next game, I would tell the vendors to add more Food stalls because Food brought in the most revenue at 4,293 dollars, which was about 50 percent of the total revenue. Every other category was much smaller, with Merch at 1,771.50 dollars and RainGear at only 901.50 dollars. This shows that Food was the most popular category. Since half of the money was already coming from Food, adding more Food stalls could let the vendors sell to more people during the game.

b) Q6 is the least trustworthy because V-18 was missing from the vendor information. Because of this, 108 orders and 2,349 dollars in revenue were put under Unknown. The revenue totals are still correct since the row count and total did not change after the merge, but we do not know the actual vendor for those orders. This is a problem when the point of the pivot table is to compare vendors.
